In [1]:
import json
import os
import re
from bs4 import BeautifulSoup

In [2]:

#Läs in JSON-filen jag skapade  
with open("data\matematik.json", "r", encoding="utf-8") as f:
    data = json.load(f)
    
subject = data.get("subject", {})
chunks = []

# Skriv ut nycklar för att dubbelkolla struktur
print(data.keys())
print(subject.keys())

dict_keys(['apiPublisher', 'apiName', 'apiVersion', 'apiReleased', 'apiStatus', 'apiDocumentation', 'dataOrigin', 'calledMethod', 'totalElements', 'processingTime', 'startedCaching', 'codeParam', 'dateParam', 'subject'])
dict_keys(['centralCntHeading', 'centralContents', 'code', 'description', 'designation', 'gradeScale', 'knowledgeReqsHeading', 'knowledgeRequirements', 'modifiedDate', 'name', 'nameHeading', 'purpose', 'purposeHeading', 'schoolTypes', 'skolfsAndring', 'skolfsGrund', 'startDate', 'typeOfSyllabus', 'version', 'versionInfo'])


<>:2: SyntaxWarning: invalid escape sequence '\m'
<>:2: SyntaxWarning: invalid escape sequence '\m'
C:\Users\daaro\AppData\Local\Temp\ipykernel_2140\1145179154.py:2: SyntaxWarning: invalid escape sequence '\m'
  with open("data\matematik.json", "r", encoding="utf-8") as f:


In [3]:
#Grundläggande metadata
subject_name = subject.get("name", "Okänt ämne")
subject_code = subject.get("code", "Okänd kod")
subject_version = subject.get("version", "Okänd version")

In [4]:
#Funktion för att ta bort HTML-taggar och returnera ren text
def clean_html(html_text):
    if not html_text:
        return ""

    soup = BeautifulSoup(str(html_text), "html.parser")
    text = soup.get_text(separator=" ", strip=True)
    return text

In [5]:
#Funktion för att dela upp text utifrån h4-taggar 
def split_text(html_text):
    if not html_text:
        return []

    soup = BeautifulSoup(str(html_text), "html.parser")

    results = []
    current_h4 = None

    for tag in soup.find_all(["h4", "ul", "p"]):
        if tag.name == "h4":
            current_h4 = tag.get_text(" ", strip=True)
        
        elif tag.name == "ul" and current_h4:
            items = [li.get_text(" ", strip=True) for li in tag.find_all("li")]
            for item in items: 
                results.append({
                    "subheading": current_h4,
                    "text": item
                })

        elif tag.name == "p" and current_h4 :
            p_text = tag.get_text(" ", strip=True)
            if p_text:
                results.append({
                    "subheading": current_h4,
                    "text": p_text
                })
        
    return results

In [6]:
central_contents = subject.get('centralContents', []) 

for item in central_contents:
    year = item.get("year")
    html_text = item.get("text", "")
    split_parts = split_text(html_text)

    for part in split_parts:
        chunks.append({
            "subject": subject_name,
            "subject_code": subject_code,
            "version": subject_version,
            "section": "centralt_innehall",
            "year": year,
            "subheading": part["subheading"],
            "heading": f"{part['subheading']} ({year})",
            "text": part["text"]
        })

In [7]:
# 1. Syfte
syfte_text = clean_html(subject.get("purpose"))
if syfte_text:
    chunks.append({
        "subject": subject_name,
        "subject_code": subject_code,
        "version": subject_version,
        "section": "syfte",
        "year": None,
        "heading": "Syfte",
        "text": syfte_text
    })

In [ ]:
# 2. Centralt innehåll
central_contents = subject.get("centralContents", [])
print(f"Hittade {len(central_contents)} centralt innehåll-objekt")

for i, item in enumerate(central_contents):
    year = item.get("year")
    html_text = item.get("text", "")

    split_parts = split_text(html_text)

    # Om det finns h4-rubriker: skapa ett chunk per underrubrik
    if split_parts:
        for part in split_parts:
            chunks.append({
                "subject": subject_name,
                "subject_code": subject_code,
                "version": subject_version,
                "section": "centralt_innehall",
                "year": year,
                "subheading": part["subheading"],
                "heading": f"{part['subheading']} ({year})",
                "text": f"{part['subheading']} för årskurs {year}. {part['text']}"
            })
    else:
        # Fallback om texten saknar h4-rubriker
        text = clean_html(html_text)
        if text:
            chunks.append({
                "subject": subject_name,
                "subject_code": subject_code,
                "version": subject_version,
                "section": "centralt_innehall",
                "year": year,
                "subheading": None,
                "heading": f"Centralt innehåll {year}",
                "text": f"Centralt innehåll {year}. {text}"
            })

Hittade 3 centralt innehåll-objekt


In [9]:
def split_knowledge_requirements(html_text):
    if not html_text:
        return []

    soup = BeautifulSoup(str(html_text), "html.parser")
    paragraphs = []

    for p in soup.find_all("p"):
        text = p.get_text(" ", strip=True)
        if text:
            paragraphs.append(text)

    return paragraphs

In [10]:
# 3. Kunskapskrav / betygskriterier
knowledge_reqs = subject.get("knowledgeRequirements", [])

for item in knowledge_reqs:
    year = item.get("year")
    grade_step = item.get("gradeStep", None)
    heading = clean_html(item.get("heading", "")) if item.get("heading") else f"Kunskapskrav {year}"
    
    parts = split_knowledge_requirements(item.get("text", ""))

    for idx, part in enumerate(parts, 1):
        chunks.append({
            "subject": subject_name,
            "subject_code": subject_code,
            "version": subject_version,
            "section": "kunskapskrav",
            "year": year,
            "grade_step": grade_step,
            "heading": f"{heading} - del {idx}",
            "text": f"Kunskapskrav för årskurs {year}. {part}"
        })

In [11]:
#Spara resultatet 
os.makedirs("data", exist_ok=True)

with open("data/matematik_chunks.json", "w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)

print(f"Klart! Skapade {len(chunks)} chunkar.")

Klart! Skapade 194 chunkar.
